# Coordinated Activity Detection

Find unusually synchronized content sharing in a synthetic public-post dataset.

**Safety and scope:** This notebook uses deterministic synthetic data and makes no network requests. Its results are analytical leads, not attribution or identity claims.

## Goal

Generate transparent coordination leads without inferring identity, intent, or authenticity from timing alone.


## Setup

The workflow runs offline with NumPy and Pandas. Parameters and source-like fields are visible so the analysis can be reviewed and rerun.

### Key Assumptions

- All records are synthetic and contain no real people or infrastructure.
- Scores prioritize review; they do not prove ownership, intent, identity, or location.
- Real use requires documented authority, provenance, source terms, and retention limits.


In [1]:
import numpy as np
import pandas as pd

SEED = 88
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
pd.set_option("display.max_colwidth", 70)


## Steps

### 1. Create bounded synthetic observations


In [2]:
post_count = 240
posts = pd.DataFrame({
    "account": rng.choice([f"acct-{index:02d}" for index in range(36)], post_count),
    "time_bucket": rng.integers(0, 48, post_count),
    "phrase_id": rng.choice([f"phrase-{index:02d}" for index in range(24)], post_count),
    "url_id": rng.choice([f"url-{index:02d}" for index in range(20)], post_count),
})
coordinated_accounts = ["acct-02", "acct-09", "acct-17", "acct-25"]
injected = pd.DataFrame({
    "account": coordinated_accounts * 3,
    "time_bucket": np.repeat([8, 21, 37], len(coordinated_accounts)),
    "phrase_id": np.repeat(["phrase-sync-a", "phrase-sync-b", "phrase-sync-c"], len(coordinated_accounts)),
    "url_id": np.repeat(["url-sync-a", "url-sync-b", "url-sync-c"], len(coordinated_accounts)),
})
posts = pd.concat([posts, injected], ignore_index=True)
print(posts.tail(8).to_string(index=False))


account  time_bucket     phrase_id     url_id
acct-02           21 phrase-sync-b url-sync-b
acct-09           21 phrase-sync-b url-sync-b
acct-17           21 phrase-sync-b url-sync-b
acct-25           21 phrase-sync-b url-sync-b
acct-02           37 phrase-sync-c url-sync-c
acct-09           37 phrase-sync-c url-sync-c
acct-17           37 phrase-sync-c url-sync-c
acct-25           37 phrase-sync-c url-sync-c


### 2. Analyze and rank the observations


In [3]:
shared = posts.merge(posts, on=["time_bucket", "phrase_id", "url_id"], suffixes=("_left", "_right"))
shared = shared[shared["account_left"] < shared["account_right"]]
pair_scores = (
    shared.groupby(["account_left", "account_right"], as_index=False)
    .size().rename(columns={"size": "synchronized_posts"})
    .sort_values("synchronized_posts", ascending=False)
)
pair_scores["review_priority"] = np.minimum(pair_scores["synchronized_posts"] / 3, 1).round(3)
print(pair_scores.head(10).to_string(index=False))


account_left account_right  synchronized_posts  review_priority
     acct-02       acct-09                   3            1.000
     acct-02       acct-17                   3            1.000
     acct-02       acct-25                   3            1.000
     acct-09       acct-17                   3            1.000
     acct-09       acct-25                   3            1.000
     acct-17       acct-25                   3            1.000
     acct-02       acct-03                   1            0.333
     acct-04       acct-25                   1            0.333
     acct-11       acct-35                   1            0.333
     acct-27       acct-35                   1            0.333


## Checks

Run deterministic integrity and reasonableness checks.


In [4]:
assert (pair_scores["account_left"] < pair_scores["account_right"]).all()
assert pair_scores["review_priority"].between(0, 1).all()
assert pair_scores["synchronized_posts"].max() >= 3
print("Checks passed; synchronization is a lead requiring content and context review.")


Checks passed; synchronization is a lead requiring content and context review.


## Next Steps

- Add source-specific rate baselines and reshare semantics.
- Review only public content allowed by source terms.
